# SPFI ×4 — training on EMPS (Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS1162/SR_amat_3/blob/main/notebooks/train_colab.ipynb)

Trains the SPFI model with the repo's `algo/` code **unchanged** (`algo.train.SPFITrainer`) on the
**train split only** of EMPS (`train.csv`). Test images are never extracted.

The result is a checkpoint file `spfi_x4_best.pt` in the same format the local app expects
(`model_state` + `hparams`), so you can use it locally for inference or for fine-tuning.

**Width comparison (C8 vs C16):** set `EMBED_DIM`, `MAX_ITERS` and `PATIENCE` as described under
*Settings*, train once per width, then compare on the validation split with `02_width_eval_val_colab.ipynb`.

**Before you start:** Runtime → Change runtime type → **GPU**. Then run the cells top to bottom.

In [ ]:
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime > Change runtime type > GPU, then rerun.")


## Settings
For a **fixed-budget width comparison** keep everything identical between runs except `EMBED_DIM`:

| setting | C8 run | C16 run |
|---|---|---|
| `EMBED_DIM` | `8` | `16` |
| `MAX_ITERS` | `5000` | `5000` |
| `MAX_HOURS` | `None` | `None` |
| `PATIENCE` | `1000` (early stopping off) | `1000` |
| `SEED` / `SPLIT_SEED` | `0` / `0` | `0` / `0` |

Both runs then see the same training patches in the same order and the same validation images.

In [ ]:
from pathlib import Path

# ---- Code (the algo/ package is used exactly as it is in the repo) ----
CODE_REPO_URL = "https://github.com/SS1162/SR_amat_3.git"
CODE_BRANCH = "main"

# ---- Data: public EMPS release (same images and train/test split as NanoVision's datasets/emps/EMPS) ----
EMPS_COMMIT = "dd8f86d714894fd08f3db80041893982ce058dd6"
EMPS_URL = f"https://codeload.github.com/by256/emps/zip/{EMPS_COMMIT}"
EMPS_SHA256 = "141462d290f9ac780bf9599ef5cb4b3e6708adfe6abafd1d4b9d6c786a6ad3f8"

WORK_DIR = Path("/content")
CODE_DIR = WORK_DIR / "SR_amat_3"
DATA_DIR = WORK_DIR / "emps"
OUTPUT_DIR = WORK_DIR / "output"

# ---- Training (defaults are the ones in algo/train.py) ----
LR = 1e-4
WEIGHT_DECAY = 1e-2
PATIENCE = 20          # early stopping: validation checks (every 250 iterations) without improvement
SEED = 0               # model init + training patch order
SPLIT_SEED = 0         # train/val image split; keep it fixed so every run has the same val images
EMBED_DIM = 48         # feature channels (model width); 48 = the default in algo/train.py

# Optional limits so the run ends before Colab disconnects. None = no limit (train until early stopping).
MAX_HOURS = 3.0
MAX_ITERS = None       # stop after exactly this many iterations (use it for fixed-budget comparisons)

RUN_TAG = f"e{EMBED_DIM}_s{SEED}"   # goes into the output file names, e.g. spfi_x4_best_e8_s0.pt

# Copy the best checkpoint to Google Drive every time it improves, so a disconnect does not lose it.
SAVE_TO_DRIVE = True
DRIVE_DIR = Path("/content/drive/MyDrive/spfi_checkpoints")


## Code and dependencies

In [ ]:
# torch / torchvision / numpy / Pillow are preinstalled on Colab; add what algo/ needs on top.
!pip install -q "pytorch_wavelets>=1.3" "PyWavelets>=1.6" "setuptools<81"


In [ ]:
import hashlib
import subprocess
import time
import urllib.request
import zipfile


def git(*args, cwd=None, attempts=3):
    """Run git, retrying network hiccups."""
    for attempt in range(1, attempts + 1):
        result = subprocess.run(["git", *args], cwd=cwd, text=True, capture_output=True)
        if result.returncode == 0:
            return result.stdout
        if attempt < attempts:
            time.sleep(5 * attempt)
    raise RuntimeError(f"git {args[0]} failed:\n{result.stderr}")


def download_emps():
    """Download the pinned public EMPS archive once (no credentials needed) and check its SHA-256."""
    archive = DATA_DIR / "emps.zip"
    if not archive.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print("Downloading EMPS...")
        urllib.request.urlretrieve(EMPS_URL, archive)
    sha = hashlib.sha256(archive.read_bytes()).hexdigest()
    if sha != EMPS_SHA256:
        print(f"WARNING: archive SHA-256 {sha} differs from the recorded {EMPS_SHA256}")
    return archive


def read_split(text):
    """Image ids (file stems) listed in train.csv / test.csv, one per line."""
    return [line.split(",")[0].strip() for line in text.splitlines() if line.strip()]


def extract_emps_split(split):
    """Extract train.csv, test.csv and only the images of `split` ('train' or 'test') into DATA_DIR/<split>/."""
    dest = DATA_DIR / split
    (dest / "images").mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(download_emps()) as zf:
        root = next(n for n in zf.namelist() if n.endswith("/train.csv"))[:-len("train.csv")]
        for name in ("train.csv", "test.csv"):
            (dest / name).write_bytes(zf.read(root + name))
        ids = read_split((dest / f"{split}.csv").read_text())
        for image_id in ids:
            (dest / "images" / f"{image_id}.png").write_bytes(zf.read(f"{root}images/{image_id}.png"))
    return dest, ids


import sys

# Always check out CODE_BRANCH, also when the folder exists from an earlier run with another branch.
if not (CODE_DIR / ".git").exists():
    git("clone", "--quiet", "--no-checkout", CODE_REPO_URL, str(CODE_DIR))
git("fetch", "--quiet", "--depth", "1", "origin", CODE_BRANCH, cwd=CODE_DIR)
git("checkout", "--quiet", "--force", "--detach", "FETCH_HEAD", cwd=CODE_DIR)
for name in [m for m in sys.modules if m == "algo" or m.startswith("algo.")]:
    del sys.modules[name]   # re-import algo/ from the branch just checked out
print("code:", CODE_BRANCH, "@", git("log", "-1", "--format=%h %s", cwd=CODE_DIR).strip())

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))   # makes `algo` importable
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Dataset
EMPS is downloaded from its public release ([by256/emps](https://github.com/by256/emps), pinned commit),
so no GitHub token is needed. Its `train.csv` / `test.csv` are identical to the ones in NanoVision's
`datasets/emps/EMPS`.

Only the images listed in `train.csv` are extracted.

In [ ]:
EMPS_DIR, train_ids = extract_emps_split("train")
test_ids = read_split((EMPS_DIR / "test.csv").read_text())

on_disk = sorted(p.stem for p in (EMPS_DIR / "images").glob("*.png"))
assert on_disk == sorted(train_ids), "images/ must hold exactly the train split"
assert not set(on_disk) & set(test_ids), "test images must not be used for training"
print(f"{len(on_disk)} train images in {EMPS_DIR / 'images'} (test split: {len(test_ids)} images, not downloaded)")


In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)


## Build the trainer
`SPFITrainer` splits the train images 80/20 into training / validation (for early stopping), so the
validation set also comes from the train split.

In [ ]:
from datetime import datetime

from algo.train import SPFITrainer

stamp = f"{RUN_TAG}_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
CKPT_DIR = WORK_DIR / "checkpoints" / stamp
RUN_DIR = WORK_DIR / "runs" / stamp       # metrics.jsonl / meta.json / samples/ (same format as the local UI)

trainer = SPFITrainer(
    emps_dir=EMPS_DIR,
    ckpt_dir=CKPT_DIR,
    lr=LR,
    wd=WEIGHT_DECAY,
    patience=PATIENCE,
    seed=SEED,
    split_seed=SPLIT_SEED,
    embed_dim=EMBED_DIM,
    run_dir=RUN_DIR,
)
print(f"embed_dim {EMBED_DIM}: {sum(p.numel() for p in trainer.model.parameters()):,} parameters")


## Train
Runs until early stopping, `MAX_HOURS` or `MAX_ITERS`, whichever comes first. A checkpoint is saved and
the model is validated every 250 iterations.

You can also press **Stop** (■) at any time: the trainer stops cleanly and you can go on to *Export*.
Running this cell again continues from where it stopped.

In [ ]:
import shutil
import threading
import time

stop_event = threading.Event()
done = threading.Event()
best_ckpt = CKPT_DIR / "spfi_x4_best.pt"


def watch(started=time.time()):
    """Stop the trainer at the time / iteration limit and mirror the best checkpoint to Drive."""
    mirrored = None
    while not done.wait(5):
        if MAX_HOURS is not None and time.time() - started >= MAX_HOURS * 3600:
            stop_event.set()
        if SAVE_TO_DRIVE and best_ckpt.exists():
            mtime = best_ckpt.stat().st_mtime
            if mtime != mirrored:
                time.sleep(1)   # let torch.save finish writing
                shutil.copy2(best_ckpt, DRIVE_DIR / f"spfi_x4_best_{stamp}.pt")
                mirrored = mtime


watcher = threading.Thread(target=watch, daemon=True)
watcher.start()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
train_started = time.time()
try:
    trainer.train(stop_event=stop_event, max_iters=MAX_ITERS)   # MAX_ITERS is exact
finally:
    train_seconds = time.time() - train_started
    done.set()
    watcher.join()
print(f"finished at iteration {trainer.iteration}, best val loss {trainer.best_val_loss:.4f}, "
      f"{train_seconds / 60:.1f} min")


## Export the weights
Saves the current weights as `spfi_x4_final_<tag>.pt` and the lowest-validation-loss weights as
`spfi_x4_best_<tag>.pt` (`<tag>` = `e<EMBED_DIM>_s<SEED>`), checks that they load the way the local app loads
them, and writes `train_info_<tag>.json` (code commit, settings, iterations, training time, peak GPU memory).
For a fixed-budget comparison, compare the **final** files: both were trained for exactly `MAX_ITERS`.

In [ ]:
import json

trainer._save_checkpoint("final")   # same format as the periodic checkpoints
final_ckpt = CKPT_DIR / "spfi_x4_final.pt"

# Same checks as the local app's inference loader (inference/runner.py::_load_model)
from algo.model import build_spfi

EXPORTS = {}
for kind, src_ckpt in (("best", best_ckpt), ("final", final_ckpt)):
    if not src_ckpt.exists():
        continue
    dst = OUTPUT_DIR / f"spfi_x4_{kind}_{RUN_TAG}.pt"
    shutil.copy2(src_ckpt, dst)
    ckpt = torch.load(dst, map_location="cpu", weights_only=True)
    hp = ckpt["hparams"]
    build_spfi(scale=hp["scale"], embed_dim=hp["embed_dim"]).load_state_dict(ckpt["model_state"])
    EXPORTS[kind] = dst
    print(f"{dst.name}: iteration {ckpt['iteration']}, best val loss so far {ckpt['best_val_loss']:.4f}, hparams {hp}")
WEIGHTS_FILE = EXPORTS.get("best", EXPORTS["final"])

train_info = {
    "run_tag": RUN_TAG,
    "code_branch": CODE_BRANCH,
    "code_commit": git("rev-parse", "HEAD", cwd=CODE_DIR).strip(),
    "embed_dim": EMBED_DIM,
    "params": sum(p.numel() for p in trainer.model.parameters()),
    "seed": SEED,
    "split_seed": SPLIT_SEED,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "patience": PATIENCE,
    "max_iters": MAX_ITERS,
    "max_hours": MAX_HOURS,
    "iterations": trainer.iteration,
    "best_val_loss": trainer.best_val_loss if trainer.best_val_loss != float("inf") else None,   # None: no validation yet
    "train_seconds": round(train_seconds, 1),
    "seconds_per_iter": round(train_seconds / max(trainer.iteration, 1), 4),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "peak_gpu_memory_mb": round(torch.cuda.max_memory_allocated() / 2**20, 1) if torch.cuda.is_available() else None,
    "torch": torch.__version__,
}
TRAIN_INFO_FILE = OUTPUT_DIR / f"train_info_{RUN_TAG}.json"
TRAIN_INFO_FILE.write_text(json.dumps(train_info, indent=2))
print(json.dumps(train_info, indent=2))

logs_zip = shutil.make_archive(str(OUTPUT_DIR / f"spfi_run_{stamp}"), "zip", RUN_DIR)
if SAVE_TO_DRIVE:
    for f in (*EXPORTS.values(), TRAIN_INFO_FILE, Path(logs_zip)):
        shutil.copy2(f, DRIVE_DIR)
    print("copied to", DRIVE_DIR)

In [ ]:
from google.colab import files

for f in EXPORTS.values():
    files.download(str(f))          # the weights files (best and final)
files.download(str(TRAIN_INFO_FILE))   # settings + training time / memory, used by the comparison notebook
files.download(logs_zip)            # optional: training curves / samples (unzip into the local runs/ folder)

## Using the weights locally
* **Inference tab** of the local app: set the model path to the downloaded `spfi_x4_best.pt`.
* **Fine-tuning:** `python -m algo.train --emps_dir <EMPS> --init finetune --weights spfi_x4_best.pt`.
* **Width comparison on the validation split:** `02_width_eval_val_colab.ipynb` (loads `spfi_x4_final_e8_s0.pt` and `spfi_x4_final_e16_s0.pt`).
* **Test PSNR / SSIM / FLOPs:** `inference_colab.ipynb` (final evaluation only; the test split stays locked).